# Landmarks com MediaPipe

O **MediaPipe**, do Google, distribui detectores de landmarks prontos, treinados com muito mais imagens do que conseguiríamos anotar, que rodam em tempo real até em CPU: 21 pontos por mão, 33 no corpo e 478 no rosto.

Com um bom detector de landmarks, muitos problemas ficam fáceis. Para reconhecer um gesto, em vez de uma CNN olhando pixels, que mudam com o fundo e a iluminação, um classificador pequeno olha os 21 pontos da mão. Neste laboratório, vemos os três detectores e o formato da saída de cada um, um exemplo completo classificando pedra, papel e tesoura pelos pontos da mão, e depois cada um monta um classificador com o seu próprio dataset.

In [ ]:
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
from mediapipe.tasks.python import BaseOptions, vision
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Os detectores do MediaPipe

Instale com `pip install -U mediapipe`. Cada detector é um arquivo `.task`, baixado uma vez, e recebe uma `mp.Image`. A `load_image` aplica a rotação guardada nos metadados das fotos de celular, para os pontos saírem no lugar certo.

Os três detectores devolvem os pontos no mesmo formato, um `NormalizedLandmark` por ponto. O $x$ e o $y$ são **normalizados** entre 0 e 1, divididos pela largura e pela altura da imagem, com $(0, 0)$ no canto superior esquerdo. O $z$ é uma estimativa da profundidade, na mesma escala do $x$ e menor quanto mais perto da câmera. A `to_pixels` converte os pontos em um array em pixels, e a `draw_landmarks` desenha os pontos ligados pelas conexões que o MediaPipe define para cada detector, com o número de cada ponto quando `numbers=True`.

In [ ]:
MODELS_DIR = "data/mediapipe"
MODEL_URLS = {
    "hand": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
    "pose": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "face": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
}


def model_path(name):
    path = os.path.join(MODELS_DIR, f"{name}.task")
    if not os.path.exists(path):
        os.makedirs(MODELS_DIR, exist_ok=True)
        urllib.request.urlretrieve(MODEL_URLS[name], path)
    return path


def load_image(source):
    if source.startswith("http"):
        path = os.path.join("data", os.path.basename(source))
        if not os.path.exists(path):
            urllib.request.urlretrieve(source, path)
        source = path
    return ImageOps.exif_transpose(Image.open(source)).convert("RGB")


def to_mp_image(image):
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(image))


def to_pixels(landmarks, image):
    W, H = image.size
    return np.array([[point.x * W, point.y * H] for point in landmarks])   # [K, 2]


def draw_landmarks(ax, points, connections, point_size=12, numbers=False):
    for connection in connections:
        ax.plot(points[[connection.start, connection.end], 0], points[[connection.start, connection.end], 1], color="lime", linewidth=1)
    ax.scatter(points[:, 0], points[:, 1], s=point_size, c="red", zorder=3)
    if numbers:
        for k, (x, y) in enumerate(points):
            ax.annotate(str(k), (x, y), xytext=(3, 3), textcoords="offset points", color="yellow", fontsize=8)

### Mãos

O detector de mãos encontra até `num_hands` mãos e devolve 21 pontos por mão. O punho é o ponto 0, e cada dedo tem quatro pontos, da base até a ponta:

| Parte | Pontos |
|---|---|
| Punho | 0 |
| Polegar | 1 (CMC), 2 (MCP), 3 (IP), 4 (ponta) |
| Indicador | 5 (MCP), 6 (PIP), 7 (DIP), 8 (ponta) |
| Médio | 9 (MCP), 10 (PIP), 11 (DIP), 12 (ponta) |
| Anelar | 13 (MCP), 14 (PIP), 15 (DIP), 16 (ponta) |
| Mínimo | 17 (MCP), 18 (PIP), 19 (DIP), 20 (ponta) |

As siglas são das articulações: a MCP liga o dedo à palma, e a PIP e a DIP são as articulações do meio e da ponta do dedo. O polegar tem uma falange a menos, e as articulações dele são a CMC, na base da palma, a MCP e a IP.

O resultado tem três listas, com um item por mão. A `hand_landmarks` traz os 21 pontos normalizados, com o $z$ medido a partir do punho. A `hand_world_landmarks` traz os mesmos pontos em metros, com origem no centro da mão. A `handedness` diz se a mão é esquerda ou direita, supondo uma imagem espelhada, como a de uma câmera frontal; em fotos comuns, o lado sai trocado.

In [ ]:
hand_detector = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(base_options=BaseOptions(model_asset_path=model_path("hand")), num_hands=2)
)
hands_image = load_image("https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg")
hands = hand_detector.detect(to_mp_image(hands_image))

print(f"{len(hands.hand_landmarks)} mãos, {len(hands.hand_landmarks[0])} pontos por mão")
print(hands.handedness[0][0])
print(hands.hand_landmarks[0][8])         # ponta do indicador, normalizada
print(hands.hand_world_landmarks[0][8])   # ponta do indicador, em metros

In [ ]:
HAND_CONNECTIONS = vision.HandLandmarksConnections.HAND_CONNECTIONS

fig, ax = plt.subplots(figsize=(7, 10))
ax.imshow(hands_image)
for landmarks in hands.hand_landmarks:
    draw_landmarks(ax, to_pixels(landmarks, hands_image), HAND_CONNECTIONS, numbers=True)
ax.axis("off")
plt.show()

### Pose

O detector de pose devolve 33 pontos do corpo. Esquerdo e direito são da pessoa: em uma foto de frente, o lado esquerdo dela aparece à direita da imagem.

| Ponto | Esquerdo | Direito |
|---|---|---|
| Nariz | 0 | 0 |
| Olho, canto interno | 1 | 4 |
| Olho | 2 | 5 |
| Olho, canto externo | 3 | 6 |
| Orelha | 7 | 8 |
| Canto da boca | 9 | 10 |
| Ombro | 11 | 12 |
| Cotovelo | 13 | 14 |
| Punho | 15 | 16 |
| Dedo mínimo | 17 | 18 |
| Dedo indicador | 19 | 20 |
| Polegar | 21 | 22 |
| Quadril | 23 | 24 |
| Joelho | 25 | 26 |
| Tornozelo | 27 | 28 |
| Calcanhar | 29 | 30 |
| Ponta do pé | 31 | 32 |

A `pose_landmarks` traz os pontos normalizados, com o $z$ medido a partir do centro dos quadris, e cada ponto tem ainda uma `visibility`, a confiança de que o ponto aparece na imagem sem estar encoberto, e uma `presence`, a confiança de que o ponto está dentro da imagem. A `pose_world_landmarks` traz os pontos em metros, com origem no centro dos quadris. Com `output_segmentation_masks=True`, o resultado traz também uma máscara da pessoa.

In [ ]:
pose_detector = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=model_path("pose")))
)
person_image = load_image("https://storage.googleapis.com/mediapipe-assets/business-person.png")
pose = pose_detector.detect(to_mp_image(person_image))

print(f"{len(pose.pose_landmarks)} pessoa, {len(pose.pose_landmarks[0])} pontos")
print(pose.pose_landmarks[0][0])   # nariz
for k, name in [(11, "ombro esquerdo"), (25, "joelho esquerdo"), (27, "tornozelo esquerdo")]:
    print(f"{name}: visibility {pose.pose_landmarks[0][k].visibility:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 10))
ax.imshow(person_image)
draw_landmarks(ax, to_pixels(pose.pose_landmarks[0], person_image), vision.PoseLandmarksConnections.POSE_LANDMARKS, numbers=True)
ax.axis("off")
plt.show()

A foto mostra só a metade de cima do corpo, mas a pose estima os 33 pontos mesmo assim, e as pernas são desenhadas abaixo da imagem. É a `visibility` que diz em quais pontos confiar.

### Rosto

O detector de rosto devolve 478 pontos: 468 formam a malha do rosto, e os 10 últimos marcam as íris, de 468 a 472 a direita e de 473 a 477 a esquerda, com o centro no primeiro de cada grupo. Os pontos de cada parte não são consecutivos, e o MediaPipe guarda as conexões de cada parte em `vision.FaceLandmarksConnections`:

| Parte | Constante | Pontos |
|---|---|---|
| Contorno do rosto | `FACE_LANDMARKS_FACE_OVAL` | 36 |
| Lábios | `FACE_LANDMARKS_LIPS` | 40 |
| Olho esquerdo, direito | `FACE_LANDMARKS_LEFT_EYE`, `FACE_LANDMARKS_RIGHT_EYE` | 16 cada |
| Sobrancelha esquerda, direita | `FACE_LANDMARKS_LEFT_EYEBROW`, `FACE_LANDMARKS_RIGHT_EYEBROW` | 10 cada |
| Íris esquerda, direita | `FACE_LANDMARKS_LEFT_IRIS`, `FACE_LANDMARKS_RIGHT_IRIS` | 4 cada, sem o centro |
| Nariz | `FACE_LANDMARKS_NOSE` | 24 |
| Todos os contornos acima | `FACE_LANDMARKS_CONTOURS` | 128 |
| Malha de triângulos | `FACE_LANDMARKS_TESSELATION` | 468 |

Alguns pontos isolados úteis: 1 é a ponta do nariz, 10 o alto da testa, 152 o queixo, 33 e 263 os cantos externos dos olhos direito e esquerdo, 61 e 291 os cantos direito e esquerdo da boca, e 468 e 473 os centros das íris direita e esquerda.

A `face_landmarks` traz os 478 pontos normalizados. Com `output_face_blendshapes=True`, a `face_blendshapes` traz 52 escores entre 0 e 1, um `_neutral` e 51 expressões, como `mouthSmileLeft` ou `eyeBlinkRight`. Com `output_facial_transformation_matrixes=True`, a `facial_transformation_matrixes` traz uma matriz $4 \times 4$ com a rotação e a posição da cabeça.

In [ ]:
face_detector = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=model_path("face")), output_face_blendshapes=True)
)
face = face_detector.detect(to_mp_image(person_image))

print(f"{len(face.face_landmarks)} rosto, {len(face.face_landmarks[0])} pontos, {len(face.face_blendshapes[0])} blendshapes")
print(face.face_landmarks[0][1])   # ponta do nariz
for category in sorted(face.face_blendshapes[0], key=lambda category: category.score, reverse=True)[:3]:
    print(f"{category.category_name}: {category.score:.2f}")

In [ ]:
KEY_POINTS = [1, 10, 152, 33, 263, 61, 291, 468, 473]   # os pontos isolados da lista acima

face_points = to_pixels(face.face_landmarks[0], person_image)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(person_image)
draw_landmarks(ax, face_points, vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS, point_size=1)
for k in KEY_POINTS:
    ax.scatter(*face_points[k], s=25, c="yellow", zorder=4)
    ax.annotate(str(k), face_points[k], xytext=(4, 4), textcoords="offset points", color="yellow", fontsize=10)
ax.set_xlim(face_points[:, 0].min() - 40, face_points[:, 0].max() + 40)   # aproxima o rosto
ax.set_ylim(face_points[:, 1].max() + 40, face_points[:, 1].min() - 40)
ax.axis("off")
plt.show()

## Exemplo: pedra, papel e tesoura

O dataset **Rock Paper Scissors**, do TensorFlow, tem 2520 imagens de treino e 372 de teste, uma pasta por classe. As mãos foram geradas por computação gráfica, e as do teste vêm de modelos diferentes das do treino.

In [ ]:
rps_dir = "data/rps"
for name in ["rps", "rps-test-set"]:
    if not os.path.exists(os.path.join(rps_dir, name)):
        os.makedirs(rps_dir, exist_ok=True)
        zip_path = os.path.join(rps_dir, f"{name}.zip")
        urllib.request.urlretrieve(f"https://storage.googleapis.com/download.tensorflow.org/data/{name}.zip", zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(rps_dir)

rps_classes = sorted(os.listdir(os.path.join(rps_dir, "rps")))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, label in zip(axes, rps_classes):
    folder = os.path.join(rps_dir, "rps", label)
    ax.imshow(load_image(os.path.join(folder, sorted(os.listdir(folder))[0])))
    ax.set_title(label)
    ax.axis("off")
plt.show()

### Extraindo os pontos

O dataset do classificador são os pontos, e não as imagens. A `extract_hand` devolve os 21 pontos da primeira mão encontrada, em pixels, para a forma da mão não sair distorcida em fotos que não são quadradas.

Usamos só o $x$ e o $y$. O $z$ é uma estimativa de profundidade feita a partir de uma única foto, pouco confiável. Neste dataset, um classificador com o $z$ acerta 100% na validação e só 77% no teste, porque o $z$ das mãos de teste é bem diferente do das mãos de treino.

A extração leva cerca de um minuto.

In [ ]:
def extract_hand(image):
    result = hand_detector.detect(to_mp_image(image))
    if not result.hand_landmarks:
        return None
    return to_pixels(result.hand_landmarks[0], image)   # [21, 2] em pixels


def extract_dataset(root):
    X, y = [], []
    for label_index, label in enumerate(rps_classes):
        folder = os.path.join(root, label)
        for name in sorted(os.listdir(folder)):
            points = extract_hand(load_image(os.path.join(folder, name)))
            if points is not None:   # sem mão detectada, a imagem fica de fora
                X.append(points)
                y.append(label_index)
    return np.stack(X), np.array(y)


X_train, y_train = extract_dataset(os.path.join(rps_dir, "rps"))
X_test, y_test = extract_dataset(os.path.join(rps_dir, "rps-test-set"))
print(X_train.shape, X_test.shape)

### Normalização

As coordenadas em pixels dependem de onde a mão está e de quão grande ela aparece, e nada disso tem a ver com o gesto. Subtraímos o punho de todos os pontos e dividimos pela maior distância de um ponto ao punho:

$$
\tilde{p}_k = \frac{p_k - p_0}{\max_j \left\| p_j - p_0 \right\|}
$$

Os 21 pontos normalizados viram um vetor de 42 valores, a entrada do classificador.

In [ ]:
def normalize_hands(X):
    X = X - X[:, :1]                                        # punho na origem
    scale = np.linalg.norm(X, axis=2).max(axis=1)           # maior distância ao punho, [N]
    return (X / scale[:, None, None]).reshape(len(X), -1)   # [N, 42]


features_train = torch.tensor(normalize_hands(X_train), dtype=torch.float32)
features_test = torch.tensor(normalize_hands(X_test), dtype=torch.float32)
print(features_train.shape)

### O classificador

Uma rede com duas camadas de 64 neurônios basta. Separamos 15% do treino para a validação e deixamos o teste, com mãos de outros modelos, para o fim.

In [ ]:
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(features_train), generator=generator)
n_val = int(0.15 * len(order))
val_idx, train_idx = order[:n_val], order[n_val:]

labels_train = torch.tensor(y_train)
train_dataloader = DataLoader(TensorDataset(features_train[train_idx], labels_train[train_idx]), batch_size=64, shuffle=True)
val_dataloader = DataLoader(TensorDataset(features_train[val_idx], labels_train[val_idx]), batch_size=256)
test_dataloader = DataLoader(TensorDataset(features_test, torch.tensor(y_test)), batch_size=256)


def accuracy(model, dataloader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in dataloader:
            correct += (model(x.to(device)).argmax(dim=1).cpu() == y).sum().item()
    return correct / len(dataloader.dataset)

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(42, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, len(rps_classes)),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
num_epochs = 50
losses, val_accuracies = [], []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for x, y in train_dataloader:
        loss = criterion(model(x.to(device)), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    losses.append(total_loss / len(train_dataloader))
    val_accuracies.append(accuracy(model, val_dataloader))

print(f"acurácia de validação {val_accuracies[-1]:.3f}")

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(range(1, num_epochs + 1), losses)
ax_loss.set_title("perda de treino")
ax_loss.set_xlabel("época")
ax_acc.plot(range(1, num_epochs + 1), val_accuracies)
ax_acc.set_title("acurácia de validação")
ax_acc.set_xlabel("época")
plt.show()

print(f"acurácia de teste {accuracy(model, test_dataloader):.3f}")

### Testando com fotos reais

O classificador foi treinado com mãos geradas por computador, mas ele só vê os pontos. Se o detector encontra os pontos de uma mão real, o classificador funciona do mesmo jeito. A `classify_hand` recebe o caminho ou a URL de uma foto, desenha os pontos e mostra a probabilidade de cada classe.

In [ ]:
def classify_hand(source):
    image = load_image(source)
    points = extract_hand(image)
    if points is None:
        print("nenhuma mão detectada")
        return

    x = torch.tensor(normalize_hands(points[None]), dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        probabilities = model(x)[0].softmax(dim=0).cpu()

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image)
    draw_landmarks(ax, points, HAND_CONNECTIONS)
    ax.set_title(", ".join(f"{label} {p:.2f}" for label, p in zip(rps_classes, probabilities)))
    ax.axis("off")
    plt.show()


classify_hand("https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg")

## O seu classificador

Agora cada um monta um classificador sobre landmarks para um problema seu, do começo ao fim. O código é por sua conta; as funções do exemplo podem ser reaproveitadas e adaptadas.

### Exercício: o problema

Escolha um problema com 3 a 6 classes que dependa da forma da mão, do corpo ou do rosto. Algumas ideias: letras estáticas de Libras, números com os dedos, posturas de yoga, postura sentada correta ou curvada, expressões faciais. Evite classes que dependem de movimento, como acenar: uma foto não registra movimento.

*Descreva aqui o problema e as classes.*

### Exercício: as fotos e os pontos

Fotografe de 40 a 60 imagens por classe, em `data/meu_classificador/<classe>/`. Varie a distância, o ângulo, o fundo e a iluminação, e, se possível, fotografe mais de uma pessoa. Extraia os pontos com o detector adequado e confira, desenhando os pontos sobre algumas fotos. Em quantas fotos o detector falhou?

In [ ]:
# extraia os pontos de todas as fotos e desenhe alguns exemplos de cada classe

### Exercício: a normalização

Adapte a normalização ao seu problema. Qual ponto serve de origem e qual distância serve de escala? No corpo, o centro dos quadris e a largura dos ombros são escolhas comuns. Vale a pena girar os pontos para eliminar a inclinação, ou a inclinação faz parte da classe, como em um joinha para cima e para baixo?

In [ ]:
# escreva a normalização e monte a entrada do classificador

### Exercício: treino e avaliação

Separe treino, validação e teste. Fotos tiradas em sequência são quase iguais, e se uma cai no treino e a vizinha no teste, o teste fica fácil demais. Se fotografou mais de uma pessoa, deixe uma pessoa inteira para o teste. Treine, mostre a acurácia de teste e algumas fotos que o modelo errou. Teste também com fotos novas e com uma foto fora das classes: o que o modelo responde?

In [ ]:
# separe os dados, treine e avalie no teste

*Comente aqui os resultados: quais classes o modelo confunde, e os erros vêm do classificador ou do detector?*